## Introduction
*Pending*

In [2]:
import pandas as pd
from pathlib import Path

## Phase 1: Initial Reconnaisance
Seeks to answer the following questions:
- Tidy or not?
- What are the entities, attributes (cols), observations (rows)?
- What variable types do I have?
- What's the rough shape and scale?
- Are there obvious problems?

In [80]:
#load arrivals datset
arrivals_path = Path("../data/Raw/Arrivals_AA_DFW_2016_2025.csv")
arrivals = pd.read_csv(arrivals_path, skiprows=7)

arrivals.head()

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Origin Airport,Scheduled Arrival Time,Actual Arrival Time,Scheduled Elapsed Time (Minutes),Actual Elapsed Time (Minutes),Arrival Delay (Minutes),Wheels-on Time,Taxi-In time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
0,AA,01/01/2016,5.0,N439AA,OKC,09:35,09:32,66.0,65.0,-3.0,09:17,15.0,0.0,0.0,0.0,0.0,0.0
1,AA,01/01/2016,6.0,N367AA,OGG,05:07,04:28,427.0,388.0,-39.0,04:24,4.0,0.0,0.0,0.0,0.0,0.0
2,AA,01/01/2016,8.0,N383AA,HNL,06:03,05:18,443.0,405.0,-45.0,05:14,4.0,0.0,0.0,0.0,0.0,0.0
3,AA,01/01/2016,25.0,N3KYAA,PHL,20:44,20:40,239.0,236.0,-4.0,20:34,6.0,0.0,0.0,0.0,0.0,0.0
4,AA,01/01/2016,36.0,N881AA,SFO,13:51,13:28,211.0,191.0,-23.0,13:21,7.0,0.0,0.0,0.0,0.0,0.0


In [81]:
#load departures datset
departures_path = Path("../data/Raw/Departures_AA_DFW_2016_2025.csv")
departures = pd.read_csv(departures_path, skiprows=7)

departures.head()

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
0,AA,01/01/2016,5.0,N361AA,HNL,12:25,14:26,516.0,529.0,121.0,14:45,19.0,121.0,0.0,13.0,0.0,0.0
1,AA,01/01/2016,7.0,N367AA,OGG,09:05,09:06,509.0,512.0,1.0,09:29,23.0,0.0,0.0,0.0,0.0,0.0
2,AA,01/01/2016,25.0,N3KYAA,PHL,12:50,12:43,185.0,161.0,-7.0,12:55,12.0,0.0,0.0,0.0,0.0,0.0
3,AA,01/01/2016,37.0,N879AA,SFO,18:45,19:04,237.0,259.0,19.0,19:22,18.0,19.0,0.0,22.0,0.0,0.0
4,AA,01/01/2016,43.0,N4YEAA,DTW,10:45,10:41,158.0,157.0,-4.0,11:04,23.0,0.0,0.0,0.0,0.0,0.0


In [82]:
departures.columns

Index(['Carrier Code', 'Date (MM/DD/YYYY)', 'Flight Number', 'Tail Number',
       'Destination Airport', 'Scheduled departure time',
       'Actual departure time', 'Scheduled elapsed time (Minutes)',
       'Actual elapsed time (Minutes)', 'Departure delay (Minutes)',
       'Wheels-off time', 'Taxi-Out time (Minutes)', 'Delay Carrier (Minutes)',
       'Delay Weather (Minutes)', 'Delay National Aviation System (Minutes)',
       'Delay Security (Minutes)', 'Delay Late Aircraft Arrival (Minutes)'],
      dtype='object')

In [85]:
# To merge arrivals and departures, change column names to a common scheme
departures_mod = departures.assign(
    flight_event="Departure", 
    scheduled_time=departures["Scheduled departure time"],
    actual_time=departures["Actual departure time"],
    wheels_time=departures["Wheels-off time"],      
    taxi_time=departures["Taxi-Out time (Minutes)"],
    delay=departures['Departure delay (Minutes)']
)

arrivals_mod = arrivals.assign(
    flight_event="Arrivals", 
    scheduled_time=arrivals["Scheduled Arrival Time"],
    actual_time=arrivals["Actual Arrival Time"],
    wheels_time=arrivals["Wheels-on Time"],      
    taxi_time=arrivals["Taxi-In time (Minutes)"],
    delay=arrivals['Arrival Delay (Minutes)']
)



In [86]:
# Drop the columns that have been renamed
dep_drop_cols = [
    "Scheduled departure time",
    "Actual departure time",
    "Taxi-Out time (Minutes)",
    "Wheels-off time",
    'Departure delay (Minutes)'
]

arr_drop_cols = [
    "Scheduled Arrival Time",
    "Actual Arrival Time",
    "Taxi-In time (Minutes)",
    "Wheels-On time",
    'Arrival Delay (Minutes)'
]

departures_mod = departures_mod.drop(columns=[c for c in dep_drop_cols if c in departures_mod.columns])
arrivals_mod= departures_mod.drop(columns=[c for c in arr_drop_cols if c in departures_mod.columns])


In [87]:
#Concatenate the two data frames
combined = pd.concat([departures_mod, arrivals_mod], ignore_index=True, sort=False)
combined.head()

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes),flight_event,scheduled_time,actual_time,wheels_time,taxi_time,delay
0,AA,01/01/2016,5.0,N361AA,HNL,516.0,529.0,121.0,0.0,13.0,0.0,0.0,Departure,12:25,14:26,14:45,19.0,121.0
1,AA,01/01/2016,7.0,N367AA,OGG,509.0,512.0,0.0,0.0,0.0,0.0,0.0,Departure,09:05,09:06,09:29,23.0,1.0
2,AA,01/01/2016,25.0,N3KYAA,PHL,185.0,161.0,0.0,0.0,0.0,0.0,0.0,Departure,12:50,12:43,12:55,12.0,-7.0
3,AA,01/01/2016,37.0,N879AA,SFO,237.0,259.0,19.0,0.0,22.0,0.0,0.0,Departure,18:45,19:04,19:22,18.0,19.0
4,AA,01/01/2016,43.0,N4YEAA,DTW,158.0,157.0,0.0,0.0,0.0,0.0,0.0,Departure,10:45,10:41,11:04,23.0,-4.0


In [88]:
combined.tail()

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes),flight_event,scheduled_time,actual_time,wheels_time,taxi_time,delay
2764935,AA,12/31/2024,3267.0,N744P,BHM,103.0,92.0,0.0,0.0,0.0,0.0,0.0,Departure,14:55,14:48,15:03,15.0,-7.0
2764936,AA,12/31/2024,3279.0,N711UW,ELP,118.0,105.0,0.0,0.0,0.0,0.0,0.0,Departure,12:26,12:23,12:38,15.0,-3.0
2764937,AA,12/31/2024,3287.0,N994NN,BOS,217.0,207.0,0.0,0.0,0.0,0.0,0.0,Departure,07:05,07:06,07:28,22.0,1.0
2764938,AA,12/31/2024,3295.0,N198UW,LAX,204.0,209.0,0.0,0.0,0.0,0.0,0.0,Departure,13:15,13:16,13:31,15.0,1.0
2764939,SOURCE: Bureau of Transportation Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Departure,NaN,NaN,NaN,NaN,NaN


In [89]:
combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2764940 entries, 0 to 2764939
Data columns (total 18 columns):
 #   Column                                    Dtype  
---  ------                                    -----  
 0   Carrier Code                              object 
 1   Date (MM/DD/YYYY)                         object 
 2   Flight Number                             float64
 3   Tail Number                               object 
 4   Destination Airport                       object 
 5   Scheduled elapsed time (Minutes)          float64
 6   Actual elapsed time (Minutes)             float64
 7   Delay Carrier (Minutes)                   float64
 8   Delay Weather (Minutes)                   float64
 9   Delay National Aviation System (Minutes)  float64
 10  Delay Security (Minutes)                  float64
 11  Delay Late Aircraft Arrival (Minutes)     float64
 12  flight_event                              object 
 13  scheduled_time                            object 
 14  ac

In [90]:
#Drop rows with missing values
combined.dropna(inplace=True)

In [91]:
#Ensure we only have "AA" as the carrier
print("Arrivals Carrier Codes:",combined["Carrier Code"].value_counts())

Arrivals Carrier Codes: Carrier Code
AA    2760362
Name: count, dtype: int64


In [92]:
#Drop Carrier Code and Tail Number Columns since those are not needed for this analysis
combined.drop(columns=["Carrier Code", "Tail Number"],inplace=True)

In [105]:
# Combine the time columns with the date to make valide datetime objects

#Change the Date column to datetime objects
combined["Date (MM/DD/YYYY)"] = pd.to_datetime(combined["Date (MM/DD/YYYY)"])

combined_mod = combined.copy() #Copying to prevent overwriting the original

tcols = ["scheduled_time", "actual_time", "wheels_time"]
for tcol in tcols:
    combined_mod[tcol + "_dt"] = pd.to_datetime(
        combined_mod["Date (MM/DD/YYYY)"].dt.strftime("%Y-%m-%d") + " " + combined_mod[tcol].astype(str),
        format="%Y-%m-%d %H:%M",
        errors="coerce" #Deal with data not in ##:## by replacing with NaN
    )
# Since the above might have resulted in NaNs, drop them again
combined_mod.dropna(inplace=True)

# # Drop the appropriate columns now that they are not needed
combined_mod.drop(columns=tcols, inplace=True)
combined_mod.drop(columns=["Date (MM/DD/YYYY)"], inplace=True)

In [107]:
combined_mod.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2759744 entries, 0 to 2764938
Data columns (total 15 columns):
 #   Column                                    Dtype         
---  ------                                    -----         
 0   Flight Number                             float64       
 1   Destination Airport                       object        
 2   Scheduled elapsed time (Minutes)          float64       
 3   Actual elapsed time (Minutes)             float64       
 4   Delay Carrier (Minutes)                   float64       
 5   Delay Weather (Minutes)                   float64       
 6   Delay National Aviation System (Minutes)  float64       
 7   Delay Security (Minutes)                  float64       
 8   Delay Late Aircraft Arrival (Minutes)     float64       
 9   flight_event                              object        
 10  taxi_time                                 float64       
 11  delay                                     float64       
 12  scheduled_time_dt  

### Tidiness
This dataset is tidy. Each variable forms one column, and each observation forms one row. We have further compressed the columns by combining dates with time-of-day columns

In [108]:
# Missing Data per column
combined.isna().sum()

Date (MM/DD/YYYY)                           0
Flight Number                               0
Destination Airport                         0
Scheduled elapsed time (Minutes)            0
Actual elapsed time (Minutes)               0
Delay Carrier (Minutes)                     0
Delay Weather (Minutes)                     0
Delay National Aviation System (Minutes)    0
Delay Security (Minutes)                    0
Delay Late Aircraft Arrival (Minutes)       0
flight_event                                0
scheduled_time                              0
actual_time                                 0
wheels_time                                 0
taxi_time                                   0
delay                                       0
dtype: int64